<a href="https://colab.research.google.com/github/vladimirvysotsky149/hse_ml/blob/main/hw2_losses_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Цель работы
Целью задания является реализация и анализ базового цикла решения задачи регрессии на табличных данных. В процессе выполнения необходимо:

- реализовать процедуру обучения и оценки модели без утечек данных;
- рассчитать и интерпретировать показатели качества MAE, RMSE, R² (при необходимости — RMSLE);
- исследовать влияние выбора функции потерь на характер и распределение ошибок модели.

## 2. Выбор датасета

Для анализа выбираем датасет raw_winequality_red из набора mnemoraorg/wine-quality-6k4

## 3. Подготовка данных

3.1. Загрузим необходимые библиотеки и датасет

In [ ]:
%pip install -q datasets seaborn pandas matplotlib scikit-learn numpy

In [ ]:
from datasets import load_dataset
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sys, os, warnings

# Подавление предупреждений (чтобы не засоряли вывод)
for warn in [UserWarning, FutureWarning]:
    warnings.filterwarnings("ignore", category=warn)

In [ ]:
dataset_dict = load_dataset("mnemoraorg/wine-quality-6k4", data_files="raw_winequality_red.csv", sep=';')

my_dataframe = dataset_dict["train"].to_pandas()

# Проверим размер и список колонок.
print("\nРазмер таблицы (строки, колонки):", my_dataframe.shape)
print("Колонки:", sorted(my_dataframe.columns.tolist()))
print("\nПервые строки:")
my_dataframe.head()

3.2. Разбиваем набор на обучающую, валидационную и тестовую части в соотношении 60 / 20 / 20

In [ ]:
# ============================================================
# Удаляем дубликаты и пересобираем сплиты
# ============================================================

from sklearn.model_selection import train_test_split

n_before = len(my_dataframe)
n_duplicates = my_dataframe.duplicated().sum()

print(f"Размер до удаления дубликатов: {my_dataframe.shape}")
print(f"Число полностью дублирующихся строк: {n_duplicates}")

my_dataframe = my_dataframe.drop_duplicates().reset_index(drop=True)

n_after = len(my_dataframe)
print(f"Размер после удаления дубликатов: {my_dataframe.shape}")
print(f"Фактически удалено строк: {n_before - n_after}")

target_col = "quality"

drop_cols = [target_col]
feature_cols = [c for c in my_dataframe.columns if c not in drop_cols]
target_vector = my_dataframe[target_col].copy()

X = my_dataframe[feature_cols]
y = my_dataframe[target_col]

print("\nЧисло признаков:", X.shape[1])
print("Список признаков:", feature_cols)

# 20% в тест
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=111,
)

# Из оставшихся 80% -> 60% train, 20% val
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=111,
)  # 0.25 * 0.8 = 0.2

print("\nTrain:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape,   y_val.shape)
print("Test: ", X_test.shape,  y_test.shape)

Разведочный анализ:

In [ ]:
cols_for_stats = feature_cols + [target_col]
print(f"\nРаспределение ({target_col}):\n")
print(my_dataframe[target_col].value_counts().round(3))

print(y_test.value_counts().round(3))

my_dataframe[cols_for_stats].describe().T

3.3. Предобрабтка данных

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()

features_numeric_columns = (
    my_dataframe[feature_cols]
    .select_dtypes(include="number")
    .columns
    .tolist()
)

# Итоговый трансформер
preprocessor = ColumnTransformer(
    transformers=[
        ("scale_numeric", numeric_transformer, features_numeric_columns),
    ],
    remainder="drop"
)

transformed_train = preprocessor.fit_transform(X_train)  # обучили трансформеры на train и применили
print("\nФорма данных до трансформации:", X_train.shape)
print("Форма данных после трансформации:", transformed_train.shape)

### 4. Построение бейзлайн-моделей

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

results = {}

def log_results(model_name: str, split: str, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    if model_name not in results:
        results[model_name] = {}

    results[model_name][split] = {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    }

def print_metrics(split_name: str, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"=== {split_name} ===")
    print(f"MSE : {mse:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE : {mae:.2f}")
    print(f"R2  : {r2:.4f}")
    print()

# ============================================================
# Бейзлайн: предсказываем качество по train
# ============================================================

baseline_name = "Baseline: mean(quality)"

# среднее только по train
y_train_mean = y_train.mean()
y_train_median = y_train.median()
print(f"Среднее {target_col} на train: {y_train_mean:.2f}")


# предсказания: одно и то же число для всех объектов
y_train_pred = np.full_like(y_train, fill_value=y_train_mean, dtype=float)
y_val_pred   = np.full_like(y_val,   fill_value=y_train_mean, dtype=float)
#y_test_pred  = np.full_like(y_test,  fill_value=y_train_mean, dtype=float)

# логируем результаты
log_results(baseline_name, "train", y_train, y_train_pred)
log_results(baseline_name, "val",   y_val,   y_val_pred)
#log_results(baseline_name, "test",  y_test,  y_test_pred)

# печатаем метрики
print("Модель:", baseline_name)
print_metrics("Train", y_train, y_train_pred)
print_metrics("Val",   y_val,   y_val_pred)
#print_metrics("Test",  y_test,  y_test_pred)

baseline_name = "Baseline: median(quality)"

# предсказания: одно и то же число для всех объектов
y_train_pred = np.full_like(y_train, fill_value=y_train_median, dtype=float)
y_val_pred   = np.full_like(y_val,   fill_value=y_train_median, dtype=float)
#y_test_pred  = np.full_like(y_test,  fill_value=y_train_median, dtype=float)

# логируем результаты
log_results(baseline_name, "train", y_train, y_train_pred)
log_results(baseline_name, "val",   y_val,   y_val_pred)
#log_results(baseline_name, "test",  y_test,  y_test_pred)

# печатаем метрики
print(f"Среднее {target_col} на train: {y_train_median:.2f}")
print("Модель:", baseline_name)
print_metrics("Train", y_train, y_train_pred)
print_metrics("Val",   y_val,   y_val_pred)
#print_metrics("Test",  y_test,  y_test_pred)

Получаем лучшее качество на предсказании средним значением, тк наше распределение не симметричное (предсказание медианой дало отрицательный результат для R2). Медианная модель делает меньше "средних" ошибок (лучше MAE), но допускает несколько больших ошибок (хуже MSE/RMSE).

### 4. Обучение линейной модели c различными функциями потерь

4.1. Модель LinearRegression с MSE

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDRegressor

linreg_sgd_name = "Linear Regression (SGD, MSE)"

linreg_sgd_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SGDRegressor(
        loss='squared_error',  # MSE (значение по умолчанию)
        penalty=None,          # без регуляризации
        max_iter=1000,         # максимальное количество эпох
        tol=1e-3,              # критерий остановки
        random_state=111
    ))
])

# обучение на train
linreg_sgd_pipeline.fit(X_train, y_train)

# предсказания
y_train_pred = linreg_sgd_pipeline.predict(X_train)
y_val_pred   = linreg_sgd_pipeline.predict(X_val)
#y_test_pred  = linreg_sgd_pipeline.predict(X_test)

# логируем результаты
log_results(linreg_sgd_name, "train", y_train, y_train_pred)
log_results(linreg_sgd_name, "val",   y_val,   y_val_pred)
#log_results(linreg_sgd_name, "test",  y_test,  y_test_pred)

print("Модель:", linreg_sgd_name)
print_metrics("Train", y_train, y_train_pred)
print_metrics("Val",   y_val,   y_val_pred)
#print_metrics("Test",  y_test,  y_test_pred)

Сравним результат с бейзлайнами.

Величина R2 ближе к 1, значит модель лучше обычного предсказания среднего. Значения абсолютной и квадратичной ошибок также снизились.

4.2. Построим диаграмму соответствия и график остатков.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Предсказания
y_pred = y_val_pred
y_true = y_val

plt.figure()
plt.scatter(y_pred, y_true)

# линия y = x
min_val = min(y_pred.min(), y_true.min())
max_val = max(y_pred.max(), y_true.max())
plt.plot([min_val, max_val], [min_val, max_val])

plt.xlabel("Predicted (ŷ)")
plt.ylabel("Actual (y)")
plt.title("Predicted vs Actual")
print("Диаграмма соответствия")
plt.show()

residuals = y_true - y_pred

plt.figure()
plt.scatter(y_pred, residuals)

plt.axhline(y=0)
plt.xlabel("Predicted (ŷ)")
plt.ylabel("Residuals (y - ŷ)")
plt.title("Residuals vs Predicted")
print("График остатков")
plt.show()

4.3. Модель LinearRegression с MAE

In [ ]:
linreg_mae_name = "Linear Regression (MAE)"

linreg_mae_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SGDRegressor(
        loss='epsilon_insensitive',
        epsilon=0.0,
        penalty=None,           # без регуляризации
        max_iter=1000,
        random_state=111
    ))
])

# обучение на train
linreg_mae_pipeline.fit(X_train, y_train)

# предсказания
y_train_pred = linreg_mae_pipeline.predict(X_train)
y_val_pred   = linreg_mae_pipeline.predict(X_val)
#y_test_pred  = linreg_mae_pipeline.predict(X_test)

# логируем результаты
log_results(linreg_mae_name, "train", y_train, y_train_pred)
log_results(linreg_mae_name, "val",   y_val,   y_val_pred)
#log_results(linreg_mae_name, "test",  y_test,  y_test_pred)

print("Модель:", linreg_mae_name)
print_metrics("Train", y_train, y_train_pred)
print_metrics("Val",   y_val,   y_val_pred)
#print_metrics("Test",  y_test,  y_test_pred)

4.4. Выбираем лучшую модель

In [ ]:
%pip install -q colorama

In [ ]:
from rich.console import Console
from rich.table import Table
from rich import box
import pandas as pd

console = Console()

def print_colored_results_table_rich(results_dict):
    """
    Печатает результаты в виде цветной таблицы с использованием rich
    """
    # Создаем таблицу
    table = Table(
        title="📊 Результаты моделей (Validation)",
        box=box.ROUNDED,
        show_header=True,
    )

    # Добавляем колонки (убираем колонку "Сплит", так как теперь только Val)
    table.add_column("Модель", style="white", no_wrap=True)
    table.add_column("MSE", justify="right", style="white")
    table.add_column("RMSE", justify="right", style="white")
    table.add_column("MAE", justify="right", style="white")
    table.add_column("R²", justify="right", style="white")

    # Фильтруем только validation сплиты
    val_metrics = []
    val_data = []  # Для хранения данных для таблицы

    for model_name, splits in results_dict.items():
        for split_name, metrics in splits.items():
            # Проверяем разные варианты написания "val"
            if split_name.lower() in ['val', 'validation', 'valid', 'validate']:
                val_metrics.append(metrics)
                val_data.append({
                    'model': model_name,
                    'metrics': metrics
                })

    # Проверяем, есть ли validation данные
    if not val_metrics:
        console.print("[bold red]Ошибка: Не найдены validation данные![/bold red]")
        console.print("Доступные сплиты:", [s for splits in results_dict.values() for s in splits.keys()])
        return

    # Находим лучшие значения только среди validation данных
    best_mse = min(m['mse'] for m in val_metrics)
    best_rmse = min(m['rmse'] for m in val_metrics)
    best_mae = min(m['mae'] for m in val_metrics)
    best_r2 = max(m['r2'] for m in val_metrics)

    worst_mse = max(m['mse'] for m in val_metrics)
    worst_rmse = max(m['rmse'] for m in val_metrics)
    worst_mae = max(m['mae'] for m in val_metrics)
    worst_r2 = min(m['r2'] for m in val_metrics)

    # Заполняем таблицу только validation данными
    for item in val_data:
        model_name = item['model']
        metrics = item['metrics']

        # Определяем цвета для каждого значения
        # MSE: меньше = лучше
        mse_style = "bold green" if metrics['mse'] == best_mse else "bold red" if metrics['mse'] == worst_mse else "white"
        mae_style = "bold green" if metrics['mae'] == best_mae else "bold red" if metrics['mae'] == worst_mae else "white"
        r2_style = "bold green" if metrics['r2'] == best_r2 else "bold red" if metrics['r2'] == worst_r2 else "white"
        rmse_style = "bold green" if metrics['rmse'] == best_rmse else "bold red" if metrics['rmse'] == worst_rmse else "white"

        table.add_row(
            model_name,
            f"[{mse_style}]{metrics['mse']:.4f}[/{mse_style}]",
            f"[{rmse_style}]{metrics['rmse']:.4f}[/{rmse_style}]",
            f"[{mae_style}]{metrics['mae']:.4f}[/{mae_style}]",
            f"[{r2_style}]{metrics['r2']:.4f}[/{r2_style}]"
        )

    # Печатаем таблицу
    console.print()
    console.print(table)
    console.print()

    # Легенда
    console.print("[bold green]Зеленый[/bold green] - лучшее значение на validation")
    console.print("[red]Красный[/red] - худшее значение на validation")


# Используем rich версию (работает в Colab)
print_colored_results_table_rich(results)


Модель LinearRegression с MSE показала лучшие результаты на валидирующей выборке по 2 из 3 показателей. Возьмем ее в качестве лучшей. Хотя видно, что модель Linear Regression в целом плохо подходит для данной выборки (низкий R2).



### 5. Финальная оценка

In [ ]:

# предсказания
y_train_pred = linreg_sgd_pipeline.predict(X_train)
y_test_pred  = linreg_sgd_pipeline.predict(X_test)

# логируем результаты
log_results(linreg_sgd_name, "test",  y_test,  y_test_pred)

print("Модель:", linreg_sgd_name)
print_metrics("Train", y_train, y_train_pred)
print_metrics("Test",  y_test,  y_test_pred)

### 6. Попробуем другую функцию потерь (Huber)

In [ ]:
linreg_hub_name = "Linear Regression (Huber)"

linreg_hub_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SGDRegressor(
        loss='huber',
        epsilon=1.35,
        penalty='l1', alpha=0.0001,
        max_iter=1000,
        random_state=111,

    ))
])

# обучение на train
linreg_hub_pipeline.fit(X_train, y_train)

# предсказания
y_train_pred = linreg_hub_pipeline.predict(X_train)

y_val_pred   = linreg_hub_pipeline.predict(X_val)
y_test_pred  = linreg_hub_pipeline.predict(X_test)

# логируем результаты
log_results(linreg_hub_name, "train", y_train, y_train_pred)
log_results(linreg_hub_name, "val",   y_val,   y_val_pred)
#log_results(linreg_hub_name, "test",  y_test,  y_test_pred)

print("Модель:", linreg_hub_name)
print_metrics("Train", y_train, y_train_pred)
print_metrics("Val",   y_val,   y_val_pred)
print_metrics("Test",  y_test,  y_test_pred)


Видим что получаем немного лучше результаты для R2 на валидирующей части, но на тестовом наборе улучшения нет.

### 7. Результаты

Для анализа набора данных разделили выборку на 3 части: обучающую, валидационную и тестовую. Первую использовали для обучения моделей, вторую использовали для выбора лучшей модели, третью для тестирования лучшей модели.

Полученные модели получились плохими (даже на обучающей выборке ) для предсказания качества вина. Это связано с тем что оценка это целое число от 1 до 10, те это ближе к задаче классификции.  